# Sequential Three-Way Training on Kaggle

Runs the three thesis ablation configs **back-to-back in one Kaggle session**:

1. Pretrained VGG16 baseline (`unet_vgg16`, ImageNet weights)
2. Scratch VGG16 (`unet_vgg16 --no-pretrained`) — isolates the pretraining contribution
3. `custom_lwir` (from-scratch DSConv + GroupNorm + SiLU + Transformer bottleneck + aux heads)

Each config writes to its own `runs/<model>_<loss>_<timestamp>_pid<PID>/`, so
the artifacts don't collide. Use `notebooks/05_analysis.ipynb` afterwards to
aggregate the results.

**Why sequential and not parallel?** One GPU at a time means logs are clean,
monitoring is trivial, and a crash in one run doesn't kill the others.
Wall-clock is ~3× one config — fine for an overnight run.

## Tips

- Use the **P100** accelerator if available (single P100 ≈ single T4 ×2 for FP32 single-GPU). Otherwise T4 ×1 also works; just ignore the second GPU.
- Save your **GitHub PAT** as a Kaggle Secret `github_pat` if the repo is private.
- Edit the `RUNS` list in §3 to add, remove, or reorder configs.

## 1. Environment check

In [ ]:
import os, platform, torch
print(f'Python:               {platform.python_version()}')
print(f'PyTorch:              {torch.__version__}')
print(f'CUDA available:       {torch.cuda.is_available()}')
print(f'GPU count:            {torch.cuda.device_count()}')
print(f'CUDA_VISIBLE_DEVICES: {os.environ.get("CUDA_VISIBLE_DEVICES", "unset")}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  [{i}] {name}  ({vram:.1f} GB)')
else:
    raise RuntimeError('No CUDA device — switch on the GPU in Settings.')

## 2. Clone repo + install deps + prepare data

All three steps are idempotent — re-running the notebook is cheap.

In [ ]:
REPO_URL    = 'https://github.com/whothemann/massmind-segmentation.git'  # EDIT if forked
REPO_BRANCH = 'main'
REPO_DIR    = '/kaggle/working/massmind-segmentation'

import os, subprocess
from pathlib import Path

def _get_pat() -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('github_pat').strip()
    except Exception:
        return ''

if Path(REPO_DIR).exists():
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    pat = _get_pat()
    url = REPO_URL.replace('https://', f'https://x-access-token:{pat}@') if pat else REPO_URL
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, url, REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL], check=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

In [ ]:
!pip install -q --upgrade-strategy only-if-needed -r requirements.txt
import torch
assert torch.cuda.is_available(), 'CUDA broke after pip install'
print('CUDA still works:', torch.cuda.get_device_name(0))

In [ ]:
# Idempotent: data + splits + stats. Skip-fast if already on disk.
!python scripts/download.py
!python -m src.splits
!python -m src.stats

## 3. Define the runs

`RUNS` is the list of configs to execute, in order. Each entry has a
human-readable `name` (used only in the summary at the end) and `args`
(the extra CLI flags appended to the shared hyperparameters).

**To skip a config:** comment out the entry.  
**To reorder:** rearrange the list — earliest entries run first.  
**To add seeds for variance:** duplicate an entry with a different `--seed`.

In [ ]:
# Shared hyperparameters across all runs. Probe-scale defaults; bump --epochs
# and drop --subset for the full thesis run (see §4).
SHARED_ARGS = [
    '--epochs',       '50',
    '--batch-size',   '8',
    '--num-workers',  '4',
    '--augmentation', 'A',
    '--loss',         'focal',
    '--focal-gamma',  '2.0',
    '--seed',         '42',
    '--amp',
    '--device',       'cuda',
]

# Set to a small integer for a pre-flight pass (e.g. SUBSET = 600 + EPOCHS = 10
# gives the probe-scale comparison). Set to None for the full thesis run.
SUBSET = None
if SUBSET is not None:
    SHARED_ARGS += ['--subset', str(SUBSET)]

RUNS = [
    {
        'name': 'vgg16_pretrained',
        'args': ['--model', 'unet_vgg16'],
    },
    {
        'name': 'vgg16_scratch',
        'args': ['--model', 'unet_vgg16', '--no-pretrained'],
    },
    {
        'name': 'custom_lwir',
        'args': ['--model', 'custom_lwir', '--aux-heads'],
    },
]

print(f'Plan: {len(RUNS)} run(s)')
for i, r in enumerate(RUNS, 1):
    print(f'  [{i}] {r["name"]:20s} {" ".join(r["args"])}')
print('\nShared:', ' '.join(SHARED_ARGS))

## 4. Execute the runs sequentially

Streams each subprocess's stdout to this notebook so you can monitor
live. Captures elapsed time per run and the final `runs/` directory of
each so we can summarise at the end.

In [ ]:
import subprocess, sys, time, re
from pathlib import Path

def _runs_before() -> set[Path]:
    """Snapshot of run subdirs before launching a config, so we can detect
    the new one this run created."""
    return set(Path('runs').glob('*/')) if Path('runs').exists() else set()

results = []
for i, run in enumerate(RUNS, 1):
    cmd = ['python', '-u', '-m', 'src.train', *SHARED_ARGS, *run['args']]
    print('=' * 78)
    print(f'[{i}/{len(RUNS)}]  {run["name"]}')
    print('  ' + ' '.join(cmd))
    print('=' * 78)

    before = _runs_before()
    t0 = time.time()
    # Stream output line-by-line so progress is visible in the notebook UI.
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = proc.wait()
    elapsed = time.time() - t0

    # Identify the new run directory created by this invocation.
    after = _runs_before()
    new_dirs = sorted(after - before, key=lambda p: p.stat().st_mtime, reverse=True)
    new_dir = str(new_dirs[0]) if new_dirs else '(unknown)'

    results.append({
        'name': run['name'],
        'rc': rc,
        'elapsed_min': elapsed / 60,
        'output_dir': new_dir,
    })
    print(f'\n[{i}/{len(RUNS)}]  {run["name"]} -> exit code {rc}  '
          f'({elapsed/60:.1f} min, {new_dir})\n')
    if rc != 0:
        print(f'WARNING: "{run["name"]}" failed (exit {rc}). Continuing with '
              'remaining runs; check the log above for the traceback.')

print('=' * 78)
print('All runs finished.')
print('=' * 78)

## 5. Summary

Per-run wall time, exit code, and the auto-generated output directory.
Use `notebooks/05_analysis.ipynb` for the actual comparison
(table + plots) after pulling `runs/` down from the Kaggle Output tab.

In [ ]:
print(f'{"name":20s}  {"exit":>4s}  {"time (min)":>11s}  output_dir')
print('-' * 90)
for r in results:
    print(f'{r["name"]:20s}  {r["rc"]:>4d}  {r["elapsed_min"]:>11.1f}  {r["output_dir"]}')

n_ok = sum(1 for r in results if r['rc'] == 0)
print(f'\n{n_ok}/{len(results)} runs succeeded.')

## 6. Package outputs for download

Zips the entire `runs/` directory into `/kaggle/working/runs.zip`. The
zip lands in the **Output tab** of the saved version — click "Download
All" to grab every checkpoint, metrics CSV, and config JSON in one go.

In [ ]:
import shutil
from pathlib import Path

archive = shutil.make_archive('/kaggle/working/runs', 'zip', root_dir='.', base_dir='runs')
print('Wrote archive:', archive)
print('Size (MB):   ', round(Path(archive).stat().st_size / 1e6, 1))